# Court document paragraph classification

This notebook runs the classification stage over existing per-case `paragraphs.parquet` files. For every BigQuery-selected case it reads the Parquet file from GCS, feeds numbered paragraph batches to the model, and writes `classification.parquet` into the same case directory. The result preserves all source columns and adds only `section`.

Use a CUDA-enabled Colab runtime for the default Hugging Face model. Execution is disabled until the prompt criteria and migration switches are configured.

In [ ]:
# Colab only; skip when the Poetry environment is already installed.
# %pip install -q "pyarrow>=24,<25" google-cloud-bigquery google-cloud-storage \
#     "transformers>=5.8,<6" huggingface-hub tqdm json-repair

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repository_root = None
source_root = None
for candidate in (cwd, *cwd.parents):
    candidate_source = candidate / "src"
    if (candidate_source / "document_split" / "__init__.py").exists():
        repository_root = candidate
        source_root = candidate_source
        break
    if (candidate / "document_split" / "__init__.py").exists():
        repository_root = candidate.parent
        source_root = candidate
        break
if repository_root is None or source_root is None:
    raise RuntimeError("Could not locate src/document_split")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from document_split.v2.document_classification import (
    CLASSIFICATION_CRITERIA_PLACEHOLDER,
    DEFAULT_CLASSIFICATION_CRITERIA,
    DEFAULT_DOCUMENT_CLASSIFICATION_SETTINGS,
    DOCUMENT_PART_CLASSIFICATION_PROMPT,
    DocumentClassificationSettings,
    classify_paragraph_parquet,
    run_document_classification_pipeline,
)
from document_split.v2.document_text_parsing import create_google_cloud_clients

print(f"Repository: {repository_root}")

## Classification criteria

Add the legal classification criteria below. Keep the output labels `introductory`, `reasoning`, and `operative`; `descriptive` is intentionally excluded for this criminal-law stage. The shared prompt template also lives in `src/document_split/v2/document_classification/prompts.py`.

In [ ]:
CLASSIFICATION_CRITERIA = ""  # Empty uses DEFAULT_CLASSIFICATION_CRITERIA

if not CLASSIFICATION_CRITERIA.strip():
    ACTIVE_PROMPT = DOCUMENT_PART_CLASSIFICATION_PROMPT
else:
    ACTIVE_PROMPT = DOCUMENT_PART_CLASSIFICATION_PROMPT.replace(
        DEFAULT_CLASSIFICATION_CRITERIA,
        CLASSIFICATION_CRITERIA,
    )

print(ACTIVE_PROMPT)

## Pipeline configuration

Set `DOCUMENT_IDS` to a tuple for an explicit smoke test or to `None` to process every eligible row selected by BigQuery. `RUN_PIPELINE` remains `False` by default.

In [ ]:
GCS_AUTH_MODE = "colab_secret"  # colab_secret, colab_user, or adc
COLAB_SERVICE_ACCOUNT_SECRET = "cloud_access"
PROJECT_ID = DEFAULT_DOCUMENT_CLASSIFICATION_SETTINGS.project_id
BIGQUERY_TABLE = DEFAULT_DOCUMENT_CLASSIFICATION_SETTINGS.bigquery_table
BUCKET = DEFAULT_DOCUMENT_CLASSIFICATION_SETTINGS.bucket
DOCUMENT_PREFIX = DEFAULT_DOCUMENT_CLASSIFICATION_SETTINGS.document_prefix
PROGRESS_COLUMN = "is_classified"
JUSTICE_KINDS = (2,)
DOCUMENT_IDS = (118355359,)  # Use None for all eligible cases
LIMIT = 10  # Keep a smoke-run limit; use None for a full run
TARGET_BATCH_TOKENS = 6_000
OVERLAP_TOKENS = 256
SKIP_EXISTING = True
RUN_PIPELINE = False

print(f"Input:  gs://{BUCKET}/{DOCUMENT_PREFIX}/<kind>/<doc_id>/paragraphs.parquet")
print(f"Output: gs://{BUCKET}/{DOCUMENT_PREFIX}/<kind>/<doc_id>/classification.parquet")

## Safe mock run

This executes the real batching, dictionary-response validation, and Parquet writer against an in-memory fake model. It does not connect to BigQuery or GCS and does not update production progress. The mock labels are deterministic test data, not legal classifications.

In [ ]:
import io
import json

import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from document_split.v2.document_text_parsing import PARAGRAPH_SCHEMA

RUN_MOCK = True
MOCK_DOCUMENT_ID = "mock-001"
MOCK_TEXTS = [
    "Вирок і реквізити суду",
    "Склад суду та відомості про обвинуваченого",
    "Суд дослідив матеріали кримінального провадження",
    "Суд оцінив докази та дійшов висновку",
    "Визнати особу винуватою",
    "Вирок може бути оскаржений",
]


class MockTokenizer:
    def encode(self, text, add_special_tokens=False):
        del add_special_tokens
        return text.split()


class MockClassificationModel:
    def __init__(self):
        self.calls = []

    def __call__(self, *, text, **_kwargs):
        user_prompt = text[1]["content"][0]["text"]
        encoded_ids = user_prompt.split(
            "TARGET PARAGRAPH IDS:\n", 1
        )[1].split("\n", 1)[0]
        target_ids = json.loads(encoded_ids)
        self.calls.append(target_ids)
        result = {}
        for paragraph_id in target_ids:
            if paragraph_id <= 2:
                section = "introductory"
            elif paragraph_id <= 4:
                section = "reasoning"
            else:
                section = "operative"
            result[str(paragraph_id)] = section
        return [{"generated_text": json.dumps(result)}]


if RUN_MOCK:
    mock_rows = [
        {
            "document_id": MOCK_DOCUMENT_ID,
            "paragraph_index": index,
            "paragraph_order": index,
            "numbered_text": f"[paragraph_id={index}] {value}",
            "text": value,
        }
        for index, value in enumerate(MOCK_TEXTS, start=1)
    ]
    source_buffer = io.BytesIO()
    pq.write_table(
        pa.Table.from_pylist(mock_rows, schema=PARAGRAPH_SCHEMA),
        source_buffer,
    )
    mock_model = MockClassificationModel()
    mock_settings = DocumentClassificationSettings(
        prompt="Mock classification contract test",
        target_batch_tokens=12,
        overlap_tokens=2,
        model_context_tokens=4_096,
        max_new_tokens=512,
    )
    mock_result = classify_paragraph_parquet(
        document_id=MOCK_DOCUMENT_ID,
        parquet_bytes=source_buffer.getvalue(),
        model_pipe=mock_model,
        tokenizer=MockTokenizer(),
        settings=mock_settings,
    )
    mock_output = (
        repository_root
        / "artifacts"
        / "document_classification_mock"
        / "classification.parquet"
    )
    mock_output.parent.mkdir(parents=True, exist_ok=True)
    mock_output.write_bytes(mock_result.parquet_bytes)
    mock_table = pq.read_table(mock_output)
    assert mock_table.column_names == [
        *PARAGRAPH_SCHEMA.names,
        "section",
    ]
    print(f"Mock model calls: {mock_model.calls}")
    print(f"Mock output: {mock_output}")
    display(mock_table.to_pandas())
else:
    print("Mock run disabled")

## One-time BigQuery progress-column migration

The current table has `is_parsed` but not `is_classified`. Review the SQL and set the switch to `True` once. The pipeline only updates `is_classified` after `classification.parquet` exists.

In [ ]:
APPLY_PROGRESS_COLUMN_MIGRATION = False
MIGRATION_SQL = f"""
ALTER TABLE `{BIGQUERY_TABLE}`
ADD COLUMN IF NOT EXISTS {PROGRESS_COLUMN} BOOL DEFAULT FALSE
""".strip()
print(MIGRATION_SQL)

if APPLY_PROGRESS_COLUMN_MIGRATION:
    migration_clients = create_google_cloud_clients(
        project_id=PROJECT_ID,
        auth_mode=GCS_AUTH_MODE,
        colab_service_account_secret=COLAB_SERVICE_ACCOUNT_SECRET,
    )
    migration_clients.bigquery.query(MIGRATION_SQL).result()
    print(f"Progress column {PROGRESS_COLUMN!r} is ready")
else:
    print("Migration disabled; review the SQL before enabling it.")

## Run classification

In Colab, add the service-account JSON as the `cloud_access` secret. If the model requires authentication, add `HF_TOKEN` as another Colab secret.

In [ ]:
if RUN_PIPELINE:
    if CLASSIFICATION_CRITERIA_PLACEHOLDER in ACTIVE_PROMPT:
        raise ValueError("Fill CLASSIFICATION_CRITERIA before running")

    hf_token = None
    if GCS_AUTH_MODE.startswith("colab"):
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None

    settings = DocumentClassificationSettings(
        project_id=PROJECT_ID,
        bigquery_table=BIGQUERY_TABLE,
        bucket=BUCKET,
        document_prefix=DOCUMENT_PREFIX,
        progress_column=PROGRESS_COLUMN,
        justice_kinds=JUSTICE_KINDS,
        document_ids=DOCUMENT_IDS,
        limit=LIMIT,
        skip_existing=SKIP_EXISTING,
        prompt=ACTIVE_PROMPT,
        target_batch_tokens=TARGET_BATCH_TOKENS,
        overlap_tokens=OVERLAP_TOKENS,
        auth_mode=GCS_AUTH_MODE,
        colab_service_account_secret=COLAB_SERVICE_ACCOUNT_SECRET,
    )
    result = run_document_classification_pipeline(
        settings,
        hf_token=hf_token,
    )
    print(result)
else:
    print("Pipeline disabled. Fill the criteria, apply the migration, then set RUN_PIPELINE=True.")